## 08 - Stock Price Analysis

This notebook measures stock price performance after each earnings call.

### What we measure?
- Stock price on earnings day
- Price change at +30, +60, and +90 days after earnings
- Whether the stock outperformed or underperformed the S&P 500

### Why these windows?
- +30 days: immediate market reaction
- +60 days: medium term reaction
- +90 days: full quarter reaction

### The key question
Did CEOs who sounded most confident actually deliver?
better stock performance? Or were they misleading investors?

In [1]:
# Import libraries
import sqlite3
import pandas as pd 
import numpy as np

# Connect to database
conn = sqlite3.connect("../data/earnings_research.db")

# Load our transcripts with sentiment and hedge scores
df = pd.read_sql("""
    SELECT h.ticker, h.quarter, h.date, h.period,
           h.avg_compound, h.hedge_ratio, h.sentiment_hedge_gap
    FROM hedge_analysis h
    ORDER BY h.date
""", conn)

print(f"Loaded {len(df)} earnings calls")
print(df[["ticker", "quarter", "date" ,"period"]].to_string(index=False))

Loaded 14 earnings calls
ticker quarter       date         period
  AAPL Q1_2020 2020-01-28      Pre-COVID
   JNJ Q1_2020 2020-04-14      Pre-COVID
   JPM Q1_2020 2020-04-14      Pre-COVID
  MSFT Q1_2020 2020-04-29      Pre-COVID
  AAPL Q2_2020 2020-04-30    COVID Crash
  AMZN Q1_2020 2020-04-30      Pre-COVID
   XOM Q1_2020 2020-05-01      Pre-COVID
   JPM Q2_2020 2020-07-14    COVID Crash
  MSFT Q4_2022 2022-07-26 Rate Hike Peak
  AAPL Q4_2022 2022-10-27 Rate Hike Peak
  MSFT Q2_2023 2023-01-24        AI Boom
  NVDA Q2_2023 2023-08-23        AI Boom
  AAPL Q1_2025 2025-01-30   Tariff Shock
  AMZN Q1_2025 2025-05-01   Tariff Shock


In [5]:
# Function to get stock price change after earnings call
def get_price_changes(ticker, earnings_date, conn):
    """
    Gets stock price on earnings day and % change at +30, +60, +90 days.
    """
    # Load stock price data for this ticker
    df_prices = pd.read_sql(f"""
        SELECT date, close
        FROM stock_prices
        WHERE ticker = '{ticker}'
        ORDER BY date
    """, conn)
    
    # Convert date column to datetime
    df_prices["date"] = pd.to_datetime(df_prices["date"].str[:10])
    earnings_date = pd.to_datetime(earnings_date)
    
    # Find closest trading day to earnings date
    df_prices["days_diff"] = abs((df_prices["date"] - earnings_date).dt.days)
    earnings_row = df_prices.loc[df_prices["days_diff"].idxmin()]
    earnings_price = earnings_row["close"]
    earnings_actual_date = earnings_row["date"]
    
    # Find prices at +30, +60, +90 days
    results = {"earnings_price": round(earnings_price, 2),
               "earnings_date": str(earnings_actual_date.date())}
    
    for days in [30, 60, 90]:
        target_date = earnings_date + pd.Timedelta(days=days)
        df_prices["days_diff"] = abs((df_prices["date"] - target_date).dt.days)
        future_row = df_prices.loc[df_prices["days_diff"].idxmin()]
        future_price = future_row["close"]
        pct_change = round((future_price - earnings_price) / earnings_price * 100, 2)
        results[f"return_{days}d"] = pct_change
    
    return results

# Test on Apple Q1 2020
result = get_price_changes("AAPL", "2020-01-28", conn)
print("Apple Q1 2020:")
print(f"Earnings day price: ${result['earnings_price']}")
print(f"30 day return: {result['return_30d']}%")
print(f"60 day return: {result['return_60d']}%")
print(f"90 day return: {result['return_90d']}%") 

Apple Q1 2020:
Earnings day price: $76.51
30 day return: -13.7%
60 day return: -21.83%
90 day return: -10.65%


In [6]:
# Run for all 14 earnings calls
price_results = []

for _, row in df.iterrows():
    try:
        result = get_price_changes(row["ticker"], row["date"], conn)
        
        price_results.append({
            "ticker": row["ticker"],
            "quarter": row["quarter"],
            "date": row["date"],
            "period": row["period"],
            "avg_compound": row["avg_compound"],
            "hedge_ratio": row["hedge_ratio"],
            "earnings_price": result["earnings_price"],
            "return_30d": result["return_30d"],
            "return_60d": result["return_60d"],
            "return_90d": result["return_90d"]
        })
        
        print(f"✓ {row['ticker']} {row['quarter']} → 30d: {result['return_30d']}% | 60d: {result['return_60d']}% | 90d: {result['return_90d']}%")
    
    except Exception as e:
        print(f"✗ {row['ticker']} {row['quarter']} failed: {e}")

print("\nAll done!")

✓ AAPL Q1_2020 → 30d: -13.7% | 60d: -21.83% | 90d: -10.65%
✓ JNJ Q1_2020 → 30d: 1.1% | 60d: -1.98% | 90d: 0.13%
✓ JPM Q1_2020 → 30d: -8.36% | 60d: 4.58% | 90d: 3.25%
✓ MSFT Q1_2020 → 30d: 3.57% | 60d: 12.15% | 90d: 14.18%
✓ AAPL Q2_2020 → 30d: 8.51% | 60d: 23.47% | 90d: 29.74%
✓ AMZN Q1_2020 → 30d: -1.28% | 60d: 8.34% | 90d: 22.62%
✓ XOM Q1_2020 → 30d: 9.36% | 60d: 5.67% | 90d: -1.06%
✓ JPM Q2_2020 → 30d: 4.24% | 60d: 2.91% | 90d: 5.28%
✓ MSFT Q4_2022 → 30d: 10.93% | 60d: -5.35% | 90d: -1.64%
✓ AAPL Q4_2022 → 30d: 2.46% | 60d: -10.05% | 90d: -1.87%
✓ MSFT Q2_2023 → 30d: 5.52% | 60d: 16.21% | 90d: 16.71%
✓ NVDA Q2_2023 → 30d: -11.68% | 60d: -8.78% | 90d: 6.01%
✓ AAPL Q1_2025 → 30d: 1.9% | 60d: -6.4% | 90d: -10.46%
✓ AMZN Q1_2025 → 30d: 7.79% | 60d: 15.35% | 90d: 21.03%

All done!


In [7]:
# Save to database
df_price_results = pd.DataFrame(price_results)

df_price_results.to_sql("price_analysis", conn, if_exists="replace", index=False)
conn.commit()

# Create master table combining everything
df_master = pd.read_sql("""
    SELECT p.ticker, p.quarter, p.date, p.period,
           p.avg_compound, p.hedge_ratio,
           p.earnings_price,
           p.return_30d, p.return_60d, p.return_90d,
           CASE 
               WHEN p.return_90d > 0 THEN 'Outperformed'
               ELSE 'Underperformed'
           END as performance_90d
    FROM price_analysis p
    ORDER BY p.date
""", conn)

df_master.to_sql("master_analysis", conn, if_exists="replace", index=False)
conn.commit()

print("Master analysis table created!")
print(df_master[["ticker", "quarter", "avg_compound", "hedge_ratio", "return_90d", "performance_90d"]].to_string(index=False))

Master analysis table created!
ticker quarter  avg_compound  hedge_ratio  return_90d performance_90d
  AAPL Q1_2020        0.2968       0.0075      -10.65  Underperformed
   JNJ Q1_2020        0.2844       0.0136        0.13    Outperformed
   JPM Q1_2020        0.1794       0.0139        3.25    Outperformed
  MSFT Q1_2020        0.3113       0.0091       14.18    Outperformed
  AAPL Q2_2020        0.3193       0.0105       29.74    Outperformed
  AMZN Q1_2020        0.1272       0.0127       22.62    Outperformed
   XOM Q1_2020        0.1838       0.0118       -1.06  Underperformed
   JPM Q2_2020        0.0901       0.0146        5.28    Outperformed
  MSFT Q4_2022        0.2529       0.0166       -1.64  Underperformed
  AAPL Q4_2022        0.2228       0.0141       -1.87  Underperformed
  MSFT Q2_2023        0.2492       0.0162       16.71    Outperformed
  NVDA Q2_2023        0.1511       0.0154        6.01    Outperformed
  AAPL Q1_2025        0.2703       0.0124      -10.46  Unde

In [8]:
# Close connection
conn.close()
print("Database connection closed.")

Database connection closed.
